# XGBoost 二阶近似如何计算切分增益？

**面试回答：**XGBoost 在当前预测处用梯度 g 与 Hessian h 近似损失，叶子最优权重为 -G/(H+lambda)，切分需带来超过 gamma 的正则化增益。

## 真实案例

点击率模型用曝光时长判断是否切分用户群，使用二分类 logistic 损失的 g/h。

In [1]:
import numpy as np  # 导入 NumPy 手写二阶统计量。
user=np.array(['U01','U02','U03','U04','U05','U06','U07','U08'])  # 构造用户编号。
dwell=np.array([3.,5.,7.,9.,12.,14.,16.,18.])  # 记录停留秒数。
y=np.array([0.,0.,0.,1.,1.,1.,1.,1.])  # 记录是否点击。
print('用户 | 停留秒 | 点击')  # 输出样本表头。
for n,v,c in zip(user,dwell,y):  # 展示用户事件。
    print(n,v,int(c))  # 输出一条事件。

用户 | 停留秒 | 点击
U01 3.0 0
U02 5.0 0
U03 7.0 0
U04 9.0 1
U05 12.0 1
U06 14.0 1
U07 16.0 1
U08 18.0 1


## Baseline / 基线

基线所有用户共享一个点击率预测。

In [2]:
base_probability=np.full(len(y),y.mean())  # 用全局点击率作为预测。
base_loss=float(-(y*np.log(base_probability)+(1-y)*np.log(1-base_probability)).mean())  # 计算基线交叉熵。
print('全局点击率=',round(float(y.mean()),3),'交叉熵=',round(base_loss,3))  # 输出基线。

全局点击率= 0.625 交叉熵= 0.662


In [3]:
score=np.zeros(len(y))  # 初始化当前 raw score 为零。
probability=1/(1+np.exp(-score))  # 将 raw score 转成当前概率。
g=probability-y  # 计算 logistic loss 的一阶梯度。
h=probability*(1-probability)  # 计算 logistic loss 的二阶 Hessian。
lambda_value=1.0  # 设置叶子 L2 正则。
gamma=.05  # 设置生长新叶所需最小增益。
def leaf_weight(gradient,hessian):  # 定义二阶近似下叶子最优权重。
    return -gradient.sum()/(hessian.sum()+lambda_value)  # 返回 -G/(H+lambda)。
def leaf_gain(gradient,hessian):  # 定义叶子带正则的增益项。
    return .5*gradient.sum()**2/(hessian.sum()+lambda_value)  # 返回二阶近似收益。
parent_gain=leaf_gain(g,h)  # 计算不切分父叶收益。
print('梯度:',np.round(g,3),'Hessian:',np.round(h,3))  # 输出一阶二阶中间量。
print('父叶权重=',round(float(leaf_weight(g,h)),3))  # 输出父叶最优更新。

梯度: [ 0.5  0.5  0.5 -0.5 -0.5 -0.5 -0.5 -0.5] Hessian: [0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25]
父叶权重= 0.333


In [4]:
rows=[]  # 保存每个候选切分。
for threshold in dwell[:-1]:  # 枚举停留时长阈值。
    left=dwell<=threshold  # 划分左右叶。
    gain=leaf_gain(g[left],h[left])+leaf_gain(g[~left],h[~left])-parent_gain-gamma  # 计算带 gamma 的切分增益。
    rows.append((threshold,gain,left))  # 保存候选统计。
best=max(rows,key=lambda row:row[1])  # 选择最大二阶增益切分。
threshold,gain,left=best  # 解包最优候选。
left_weight=leaf_weight(g[left],h[left])  # 计算左叶最优权重。
right_weight=leaf_weight(g[~left],h[~left])  # 计算右叶最优权重。
update=np.where(left,left_weight,right_weight)  # 为每个用户分配叶子更新。
new_probability=1/(1+np.exp(-(score+.5*update)))  # 用学习率更新点击概率。
new_loss=float(-(y*np.log(new_probability)+(1-y)*np.log(1-new_probability)).mean())  # 计算一轮后的交叉熵。
print('候选阈值/增益:',[(round(a,1),round(b,3)) for a,b,c in rows])  # 输出切分增益表。
print('最优阈值/增益/叶权重:',threshold,round(gain,3),round(left_weight,3),round(right_weight,3))  # 输出二阶切分结果。

候选阈值/增益: [(3.0, 0.292), (5.0, 0.917), (7.0, 1.815), (9.0, 1.033), (12.0, 0.482), (14.0, 0.117), (16.0, -0.071)]
最优阈值/增益/叶权重: 7.0 1.815 -0.857 1.111


## 结果解读

Hessian 是损失曲率而非样本协方差；当 H 很小时，lambda 抑制过大的叶权重。gamma 则要求切分收益超过结构复杂度成本。

In [5]:
print('用户 | 原概率 | 新概率 | 点击')  # 输出逐用户结果表头。
for n,a,b,c in zip(user,probability,new_probability,y):  # 展示一轮更新。
    print(n,round(float(a),3),round(float(b),3),int(c))  # 输出概率变化。
print('交叉熵 基线/一轮=',round(base_loss,3),round(new_loss,3))  # 输出指标比较。
print('生产差距：完整 XGBoost 还需直方图、缺失默认方向、采样、并行与时间验证。')  # 说明工程边界。

用户 | 原概率 | 新概率 | 点击
U01 0.5 0.394 0
U02 0.5 0.394 0
U03 0.5 0.394 0
U04 0.5 0.635 1
U05 0.5 0.635 1
U06 0.5 0.635 1
U07 0.5 0.635 1
U08 0.5 0.635 1
交叉熵 基线/一轮= 0.662 0.472
生产差距：完整 XGBoost 还需直方图、缺失默认方向、采样、并行与时间验证。


## 失败案例与修复

若忽略 gamma，小叶子也可能因微小噪声切分；修复是最小增益与最小 Hessian 约束。

In [6]:
tiny=rows[-1]  # 取靠近单样本叶的候选。
raw_gain=tiny[1]+gamma  # 恢复不扣 gamma 的原始增益。
print('失败：不扣gamma的小叶原始增益=',round(raw_gain,3))  # 输出可能被误接受的收益。
print('修复：扣gamma后收益=',round(tiny[1],3),'是否生长=',tiny[1]>0)  # 输出结构正则门禁。
print('真实项目还应设置 min_child_weight，避免低曲率叶子。')  # 补充生产护栏。

失败：不扣gamma的小叶原始增益= -0.021
修复：扣gamma后收益= -0.071 是否生长= False
真实项目还应设置 min_child_weight，避免低曲率叶子。


In [7]:
assert len(user)>=5  # 保护样本数。
assert np.all(h>0)  # 保护 logistic Hessian 为正。
assert new_loss<base_loss  # 保护二阶更新降低教学损失。
assert gain>0  # 保护最优切分通过 gamma 门槛。